In [32]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 113.2 MB/s eta 0:00:00


In [33]:
import fitz  # PyMuPDF

# Open the PDF file
pdf_document = fitz.open("/content/llm.pdf")

# Extract text from each page
text = ""
for page_num in range(pdf_document.page_count):
    page = pdf_document.load_page(page_num)
    text += page.get_text()

# Close the document
pdf_document.close()

# Now you can proceed with creating your tokenizer using the extracted text
# Create a character-level vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocabulary size:", vocab_size)

# Create mappings: char -> id, id -> char
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encode entire text
data = [stoi[ch] for ch in text]
print("First 20 token IDs:", data[:20])

Vocabulary size: 111
First 20 token IDs: [16, 18, 15, 16, 21, 15, 17, 20, 20, 23, 1, 31, 38, 12, 1, 17, 18, 26, 19, 24]


In [34]:
import torch
import torch.nn as nn

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=2, hidden_dim=64, num_layers=2, seq_len=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(seq_len, embed_dim)  # positional encoding
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim)
            for _ in range(num_layers)
        ])
        self.ln = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_embed(positions)
        for layer in self.layers:
            x = layer(x)
        x = self.ln(x)
        return self.fc(x)


In [35]:
import random
import torch

seq_len = 32

def get_batch(data, batch_size=4):
    x_batch, y_batch = [], []
    for _ in range(batch_size):
        start = random.randint(0, len(data)-seq_len-1)
        x = data[start:start+seq_len]
        y = data[start+1:start+seq_len+1]
        x_batch.append(x)
        y_batch.append(y)
    return torch.tensor(x_batch), torch.tensor(y_batch)


In [36]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyTransformer(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    x, y = get_batch(data)
    x, y = x.to(device), y.to(device)

    optimizer.zero_grad()
    logits = model(x)  # [batch, seq_len, vocab_size]

    loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 4.9551
Epoch 10, Loss: 4.5605
Epoch 20, Loss: 4.3359
Epoch 30, Loss: 4.1915
Epoch 40, Loss: 3.9364
Epoch 50, Loss: 3.6587
Epoch 60, Loss: 3.8569
Epoch 70, Loss: 3.7702
Epoch 80, Loss: 3.4822
Epoch 90, Loss: 3.5659


In [42]:
def generate(model, start_text="I", length=50):
    model.eval()
    idxs = [stoi[ch] for ch in start_text]
    idxs = torch.tensor(idxs, device=device, dtype=torch.long).unsqueeze(0)

    for _ in range(length):
        logits = model(idxs)
        next_token = torch.argmax(logits[0, -1]).clamp(0, vocab_size-1).long()
        idxs = torch.cat([idxs, next_token.unsqueeze(0).unsqueeze(0)], dim=1)

    return "".join([itos[i.item()] for i in idxs[0]])
